# Sentiment Analysis

PyTorch configuration

In [3]:
import torch
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Parsing the dataset that consists of labeled amazon reviews

In [5]:
list_of_strings = []
list_of_labels = []

import csv
with open('reviews_with_labels.csv') as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
      list_of_strings.append(row[1])
      list_of_labels.append(float(row[2]))

Extracting the whole vocabulary, and associating each word with a word ID (basically tokenizing each word). Transforming each sentence of the training set into a tensor of token IDs, and padding it so they all have the same length.

In [6]:
def get_dataset(list_of_strings):

    # First let's get the total set of words
    words = set()
    for sentence in list_of_strings:
        for word in sentence.split():
            words.add(word)

    vocab_size = len(words)

    # Now let's build a mapping
    sorted_list = sorted(list(words))
    word_to_int = {}
    for i, c in enumerate(sorted_list):
        word_to_int[c] = i + 1

    # Write encode() which is used to build the dataset
    def encode(sentence):
        integers = []
        for word in sentence.split():
            integers.append(word_to_int[word])
        return integers

    var_len_tensors = []
    for sentence in list_of_strings:
        var_len_tensors.append(torch.tensor(encode(sentence)))

    # Return a tuple containing the vocab size, the sequence of string converted to the corresponding
    # token ids and padded with 0 so they all have the same length, and the dictionnary mapping each
    # word to its token ID.
    return vocab_size + 1, nn.utils.rnn.pad_sequence(var_len_tensors, batch_first = True), word_to_int

In [7]:
vocab_size, training_dataset, word_to_int = get_dataset(list_of_strings)
training_labels = torch.unsqueeze(torch.tensor(list_of_labels), dim = -1)

Simple sentiment analysis model structure : 
- Embedding layer associating each token with an embedding
- Averaging of the embeddings in each sentence
- Linear layer producing a single score from this averaged embedding
- Tanh activation function, mapping the output to (-1, 1), corresponding to the sentiment labels

In [8]:
class EmotionPredictor(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_dimension: int):
        super().__init__()
        self.embedding_layer = nn.Embedding(vocabulary_size, embedding_dimension)
        self.linear_layer = nn.Linear(embedding_dimension, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        embeddings = self.embedding_layer(x)
        averaged = torch.mean(embeddings, axis = 1)
        projected = self.linear_layer(averaged)
        return self.tanh(projected)

Train the model using a standard training loop. At each iteration, we randomly shuffle the dataset and select a mini-batch of 64 examples from the full dataset. The model is then updated based only on this mini-batch, which makes training much faster and more memory-efficient.

In [36]:
embedding_dimension = 256
model = EmotionPredictor(vocab_size, embedding_dimension)
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters())

for i in range(1000):
  randperm = torch.randperm(len(training_dataset))
  training_dataset, training_labels = training_dataset[randperm], training_labels[randperm]
  mini_batch = training_dataset[:64]
  mini_batch_labels = training_labels[:64]
  pred = model(mini_batch)
  optimizer.zero_grad()
  loss = loss_function(pred, mini_batch_labels)
  if (i + 1) % 100 == 0:
    print(f"Loss at iteration no {i + 1} : {loss.item()}")
  loss.backward()
  optimizer.step()

Loss at iteration no 100 : 0.1718200147151947
Loss at iteration no 200 : 0.13049837946891785
Loss at iteration no 300 : 0.12482333928346634
Loss at iteration no 400 : 0.14610399305820465
Loss at iteration no 500 : 0.09475211054086685
Loss at iteration no 600 : 0.10392095893621445
Loss at iteration no 700 : 0.12233756482601166
Loss at iteration no 800 : 0.07497335225343704
Loss at iteration no 900 : 0.10460858792066574
Loss at iteration no 1000 : 0.1127588301897049


We can now test the model on some new sentences it has never seen before

In [37]:
example_one = "worst movie ever"
example_two = "best movie ever"
example_three = "weird but funny movie"

examples = [example_one] + [example_two] + [example_three]

# Encode these strings as numbers using the dictionary from earlier
var_len = []
for example in examples:
  int_version = []
  for word in example.split():
    int_version.append(word_to_int[word])
  var_len.append(torch.tensor(int_version))

testing_tensor = torch.nn.utils.rnn.pad_sequence(var_len, batch_first=True)

model.eval()
print(model(testing_tensor).tolist())

[[-0.9998685121536255], [0.9980419874191284], [-0.18864618241786957]]


The model outputs something very close to -1 for example one (negative), something very close to 1 for example two (positive), and something closer to 0 for example three (neutral).

This is a very simple example of a sentiment analyser model. 